In [0]:
df = spark.table("bankingdata.bronze.branch")

In [0]:
df.columns

### Handling rescue data column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_rescued = df.filter(col("_rescued_data").isNotNull())
df = df.filter(col("_rescued_data").isNull())

In [0]:
df_rescued_final = df_rescued.select(
    col("BranchID"),          
    col("_rescued_data"),       
    col("source_file"),        
    col("ingestion_time")       
)

In [0]:
df_rescued_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.rescueddata.branch_rescued")

### Schema

In [0]:
df = df.select(
    col("BranchID").cast("int").alias("BranchID"),
    col("BranchName").cast("string").alias("BranchName"),
    col("City").cast("string").alias("City"),
    col("Province").cast("string").alias("Province"),
    col("ModifiedDate").cast("timestamp").alias("ModifiedDate")
)

### Primary Key Validation

In [0]:
df = df.filter(
    col("BranchID").isNotNull() &
    (trim(col("BranchID").cast("string")) != "") &
    (col("BranchID") > 0)
)

### Business Quality Check


In [0]:
df = df.filter(
    # BranchName
    col("BranchName").isNotNull() &
    (length(trim(col("BranchName"))) > 2) &

    # City
    col("City").isNotNull() &
    (length(trim(col("City"))) > 2) &

    # Province (Canadian example)
    col("Province").isNotNull() &
    col("Province").isin("ON", "BC", "AB", "QC") &

    # ModifiedDate
    col("ModifiedDate").isNotNull() &
    (col("ModifiedDate") <= current_timestamp())
)

### Remove Duplicates

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("BranchID") \
                    .orderBy(col("ModifiedDate").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
                   .filter(col("rn") == 1) \
                   .drop("rn")

### Delta table

In [0]:
# save table
silver_table = "bankingdata.silver.branch"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)